# CommonLit Readability — DistilBERT 추론 제출 커널 (v2)
인터넷 차단 환경 → 모델은 첨부 데이터셋에서 오프라인 로드. 경로는 자동탐색하고, GPU(P100) 호환 이슈 회피 위해 CPU로 추론.

In [ ]:
import os, glob, numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 입력 트리 확인 (경로 디버그용)
print("=== /kaggle/input tree ===")
for root, dirs, files in os.walk("/kaggle/input"):
    if root.count("/") - 2 <= 3:
        print(root)
        for f in files[:6]: print("    -", f)

In [ ]:
# 모델 폴더: config.json + model.safetensors 둘 다 있는 곳
model_dirs = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/config.json", recursive=True)
              if os.path.exists(os.path.join(os.path.dirname(p), "model.safetensors"))]
MODEL_DIR = model_dirs[0]
print("MODEL_DIR =", MODEL_DIR)

# test.csv: 'excerpt' 컬럼 있는 걸 자동 선택 (대회 데이터)
test_path = None
for p in glob.glob("/kaggle/input/**/test.csv", recursive=True):
    try:
        if "excerpt" in pd.read_csv(p, nrows=1).columns:
            test_path = p; break
    except Exception:
        pass
print("test_path =", test_path)
assert test_path is not None, "test.csv(excerpt 포함)를 못 찾음"

In [ ]:
device = "cpu"   # Kaggle P100(sm_60)가 현재 torch와 비호환 -> CPU로 추론(가벼워서 충분)
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device).eval()

test = pd.read_csv(test_path)
print("test shape:", test.shape)
enc = tok(list(test["excerpt"]), truncation=True, padding="max_length", max_length=256, return_tensors="pt")

preds = []
with torch.no_grad():
    for i in range(0, len(test), 16):
        ids  = enc["input_ids"][i:i+16].to(device)
        attn = enc["attention_mask"][i:i+16].to(device)
        logits = model(input_ids=ids, attention_mask=attn).logits.squeeze(-1)
        preds.append(logits.detach().cpu().numpy().reshape(-1))
pred = np.concatenate(preds)

sub = pd.DataFrame({"id": test["id"], "target": pred})
sub.to_csv("submission.csv", index=False)
print(sub.head())
print("rows:", len(sub), "| range:", round(float(pred.min()),3), "~", round(float(pred.max()),3))